# 05B — Train One Segmentation Model

**Run this notebook once per model. Change only `CONFIG_NAME`, then run top to bottom.**

### Datasets to attach
| Dataset | Role |
|---|---|
| Your code zip (`brats-seg-code`) | Source code |
| `brats2020-2d-prepared` (output of 05A) | Prepared slices + split |

### Workflow
Run this notebook 5 times — one per config — saving each checkpoint output before changing `CONFIG_NAME`:
```
default.yaml                 → unet_baseline
unet_best_fusion_loss_aug.yaml → unet_best (optimized loss/aug)
unetpp.yaml                  → unetpp
attention_unet.yaml          → attention_unet
uncertainty_unet.yaml        → uncertainty_unet  (MC-dropout)
```

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
CODE_DATASET_SLUG = 'brats-seg-code'          # your uploaded code dataset

# Change this line to train a different model:
CONFIG_NAME = 'default.yaml'
# CONFIG_NAME = 'unet_best_fusion_loss_aug.yaml'
# CONFIG_NAME = 'unetpp.yaml'
# CONFIG_NAME = 'attention_unet.yaml'
# CONFIG_NAME = 'uncertainty_unet.yaml'

# Set to 2 for a quick smoke test. None = use the config's real epoch count.
EPOCH_OVERRIDE = None

In [ ]:
import os, shutil, subprocess, sys, json, yaml
from pathlib import Path

def disk_gb(path='/'):
    st = os.statvfs(path)
    return round(st.f_bavail * st.f_frsize / 1e9, 2)

print(f'Free disk: {disk_gb()} GB')

In [ ]:
# ── 1. Find and copy code repo ─────────────────────────────────────────────
def looks_like_repo(p):
    return (p / 'src').exists() and (p / 'scripts').exists() and (p / 'configs').exists()

repo_input = Path('/kaggle/input') / CODE_DATASET_SLUG
if not looks_like_repo(repo_input):
    for p in Path('/kaggle/input').rglob('requirements.txt'):
        if looks_like_repo(p.parent):
            repo_input = p.parent
            break

if not looks_like_repo(repo_input):
    raise FileNotFoundError(f'Cannot find repo under /kaggle/input. Set CODE_DATASET_SLUG. Checked: {repo_input}')

PROJECT = Path('/kaggle/working/project')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
shutil.copytree(
    repo_input, PROJECT,
    ignore=shutil.ignore_patterns('.git', '__pycache__', '.pytest_cache', 'data', 'outputs', 'checkpoints')
)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

def run(cmd):
    subprocess.check_call(cmd, cwd=str(PROJECT))

print('Repo:', repo_input, '→', PROJECT)

In [ ]:
# ── 2. Install missing packages only ──────────────────────────────────────
# Kaggle already has: torch, torchvision, numpy, pandas, scipy, sklearn,
#                     opencv, matplotlib, seaborn, tqdm, pyyaml, jupyter
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations'])

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Free disk: {disk_gb()} GB')

In [ ]:
# ── 3. Find prepared data ──────────────────────────────────────────────────
prepared_candidates = [
    Path('/kaggle/input/brats2020-2d-prepared'),
    Path('/kaggle/input/brats2020_2d_prepared'),
]
PREPARED = next((p for p in prepared_candidates if (p / 'data/brats2020_2d/metadata.json').exists()), None)
if PREPARED is None:
    # fallback: search
    hits = list(Path('/kaggle/input').rglob('metadata.json'))
    PREPARED = hits[0].parent.parent.parent if hits else None
if PREPARED is None:
    raise FileNotFoundError('Cannot find brats2020-2d-prepared. Attach the dataset output from notebook 05A.')

DATA_ROOT  = PREPARED / 'data/brats2020_2d'
SPLIT_FILE = PREPARED / 'configs/splits/default.json'
print(f'Prepared data: {DATA_ROOT}')

with open(DATA_ROOT / 'metadata.json') as f:
    meta = json.load(f)
with open(SPLIT_FILE) as f:
    split = json.load(f)
print(f'Slices: {meta["num_slices"]:,}  |  dtype: {meta.get("image_dtype", "?")}  |  Split: {dict((k, len(v)) for k, v in split.items())}')

In [ ]:
# ── 4. Build runtime config (patch paths to this session's locations) ──────
base_cfg_path = PROJECT / 'configs' / CONFIG_NAME
with open(base_cfg_path) as f:
    cfg = yaml.safe_load(f)

cfg['data']['data_root']  = str(DATA_ROOT)
cfg['data']['split_file'] = str(SPLIT_FILE)
cfg['checkpoint_dir']     = '/kaggle/working/checkpoints'
cfg['experiment']['output_dir'] = '/kaggle/working/outputs'

if EPOCH_OVERRIDE is not None:
    cfg['training']['epochs'] = int(EPOCH_OVERRIDE)
    cfg['training']['early_stopping_patience'] = max(2, min(int(EPOCH_OVERRIDE), 15))

runtime_cfg = Path('/kaggle/working/runtime_config.yaml')
with open(runtime_cfg, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f'Model:   {cfg["model"]["arch"]}')
print(f'Epochs:  {cfg["training"]["epochs"]}')
print(f'Batch:   {cfg["data"]["batch_size"]}')
print(f'Exp:     {cfg["experiment"]["name"]}')

In [ ]:
# ── 5. Train ───────────────────────────────────────────────────────────────
run([sys.executable, '-m', 'scripts.train', '--config', str(runtime_cfg)])

In [ ]:
# ── 6. Show training curves ────────────────────────────────────────────────
import json, matplotlib.pyplot as plt

exp_name  = cfg['experiment']['name']
hist_path = Path('/kaggle/working/outputs') / exp_name / 'history.json'

if hist_path.exists():
    with open(hist_path) as f:
        history = json.load(f)

    epochs     = [h['epoch']      for h in history]
    train_dice = [h['train_dice'] for h in history]
    val_dice   = [h['val_dice']   for h in history]
    val_loss   = [h['val_loss']   for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, train_dice, label='train'); axes[0].plot(epochs, val_dice, label='val')
    axes[0].set_title(f'{exp_name} — Dice'); axes[0].set_xlabel('Epoch'); axes[0].legend()
    axes[1].plot(epochs, val_loss, color='orange')
    axes[1].set_title('Val Loss'); axes[1].set_xlabel('Epoch')
    plt.tight_layout(); plt.show()

    best = max(history, key=lambda h: h['val_dice'])
    print(f'Best epoch {best["epoch"]}:  val_dice={best["val_dice"]:.4f}  val_loss={best["val_loss"]:.4f}')
else:
    print('history.json not found — check that trainer saves it')

In [ ]:
# ── 7. Verify checkpoint exists ────────────────────────────────────────────
ckpt = Path('/kaggle/working/checkpoints') / f'{exp_name}_best.pth'
print(f'Checkpoint: {ckpt}  exists={ckpt.exists()}')
if ckpt.exists():
    print(f'Size: {ckpt.stat().st_size / 1e6:.1f} MB')
print(f'Free disk: {disk_gb()} GB')
print()
print('NEXT STEP: Save Version → the checkpoint will be in Kaggle output.')
print('Then change CONFIG_NAME and re-run for the next model.')